In [6]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [7]:
simulator = BasicSimulator()

NUM_QUBITS = 24
ERROR_THRESHOLD = 0.10

# Custom random bit generator using a superposition state
def get_quantum_random_bit():
    qrng_qc = QuantumCircuit(1, 1)
    qrng_qc.h(0)
    qrng_qc.measure(0, 0)

    job = transpile(qrng_qc, simulator)
    result = simulator.run(job, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [8]:
# Alice generates her bits and chooses bases (0=std, 1=diag)
alice_bits = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]
alice_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)

# Encode the states
for i in range(NUM_QUBITS):
    if alice_bits[i] == 1:
        qc.x(i)
    if alice_bases[i] == 1:
        qc.h(i)

print("Alice has prepared and sent the qubits to Bob.")

Alice has prepared and sent the qubits to Bob.


In [9]:
# Bob chooses his random bases
bob_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

# Measure received qubits
for i in range(NUM_QUBITS):
    if bob_bases[i] == 1:
        qc.h(i)
    qc.measure(i, i)

job = transpile(qc, simulator)
result = simulator.run(job, shots=1).result()
counts = result.get_counts()

# Reverse the string to match indices
measured_str = list(counts.keys())[0][::-1]
bob_bits = [int(bit) for bit in measured_str]

print("Bob has finished measuring the received qubits.")

Bob has finished measuring the received qubits.


In [10]:
alice_key = []
bob_key = []

# Sift keys
for i in range(NUM_QUBITS):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

print(f"Alice bits:   {alice_bits}")
print(f"Alice bases:  {alice_bases}")
print(f"Bob bases:    {bob_bases}")
print(f"Bob bits:     {bob_bits}")
print("-" * 40)
print(f"Alice key:    {alice_key}")
print(f"Bob key:      {bob_key}")
print(f"Keys match:   {alice_key == bob_key}")

# Sacrifice half the key to check for errors
sample_size = len(alice_key) // 2
alice_sample = alice_key[:sample_size]
bob_sample = bob_key[:sample_size]

errors = sum(1 for i in range(sample_size) if alice_sample[i] != bob_sample[i])
error_rate = errors / sample_size if sample_size > 0 else 0

print("-" * 30)
print(f"Sifted key length: {len(alice_key)}")
print(f"Sample size used:  {sample_size}")
print(f"Errors found:      {errors}")
print(f"Error rate:        {error_rate*100:.2f}%")
print("-" * 30)

if error_rate > ERROR_THRESHOLD:
    print("ALERT: Attack detected! Error rate is over the 10% threshold. Key has therefore been compromised and will be discarded.")
else:
    print("Key is safe.")

Alice bits:   [0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0]
Alice bases:  [0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0]
Bob bases:    [1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0]
Bob bits:     [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0]
----------------------------------------
Alice key:    [0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0]
Bob key:      [0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0]
Keys match:   True
------------------------------
Sifted key length: 11
Sample size used:  5
Errors found:      0
Error rate:        0.00%
------------------------------
Key is safe.
